# A smoke test for classification baseline

In [1]:
from pathlib import Path
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import numpy as np


In [2]:
DATA_ROOT = Path("../brisc2025")
SPLIT_ROOT = Path("../data")

train_df = pd.read_csv(SPLIT_ROOT / "classification_train.csv")
val_df = pd.read_csv(SPLIT_ROOT / "classification_val.csv")

CLASS_NAMES = ["glioma", "meningioma", "no_tumor", "pituitary"]
CLASS_TO_INDEX = {name: index for index, name in enumerate(CLASS_NAMES)}

In [14]:
class ClassificationDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        row = self.dataframe.iloc[index]
        image_path = DATA_ROOT / row["relative_path"]

        image = Image.open(image_path).convert("RGB")
        label = CLASS_TO_INDEX[row["tumor_label"]]

        if self.transform:
            image = self.transform(image)

        return image, label

In [18]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])

In [19]:
train_dataset = ClassificationDataset(train_df,train_transform)
val_dataset = ClassificationDataset(val_df, val_transform)

train_loader = DataLoader( train_dataset, batch_size=32, shuffle=True,)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False,)


In [20]:
imgs, labels = next(iter(train_loader))

print(imgs.shape)
print(labels.shape)
print(labels[:10])

torch.Size([32, 3, 224, 224])
torch.Size([32])
tensor([3, 3, 0, 3, 2, 1, 3, 3, 0, 1])


In [21]:
# resnet 18
import torch.nn as nn
from torchvision.models import resnet18, ResNet18_Weights

weights = ResNet18_Weights.DEFAULT
model = resnet18(weights=weights)

model.fc = nn.Linear(
    model.fc.in_features,
    len(CLASS_NAMES),
)

In [22]:
device = (
    torch.device("cuda") if torch.cuda.is_available()
    else torch.device("mps") if torch.backends.mps.is_available()
    else torch.device("cpu")
)

model = model.to(device)
print("Device:", device)

Device: mps


In [23]:
imgs, labels = next(iter(train_loader))
imgs = imgs.to(device)

with torch.no_grad():
    logits = model(imgs)

print("Input:", imgs.shape)
print("Output:", logits.shape)

Input: torch.Size([32, 3, 224, 224])
Output: torch.Size([32, 4])


In [24]:
criterion = torch.nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    model.fc.parameters(),
    lr=1e-3,
    weight_decay=1e-4,
)

In [26]:
for parameter in model.parameters():
    parameter.requires_grad = False

for parameter in model.fc.parameters():
    parameter.requires_grad = True

In [27]:
from sklearn.metrics import f1_score

def run_epoch(model, loader, training):
    model.train(training)

    total_loss = 0
    all_predictions = []
    all_labels = []

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        if training:
            optimizer.zero_grad()

        with torch.set_grad_enabled(training):
            logits = model(images)
            loss = criterion(logits, labels)

            if training:
                loss.backward()
                optimizer.step()

        total_loss += loss.item() * images.size(0)
        all_predictions.extend(logits.argmax(dim=1).cpu().tolist())
        all_labels.extend(labels.cpu().tolist())

    average_loss = total_loss / len(loader.dataset)
    accuracy = (
        np.array(all_predictions) == np.array(all_labels)
    ).mean()
    macro_f1 = f1_score(
        all_labels,
        all_predictions,
        average="macro",
    )

    return average_loss, accuracy, macro_f1

In [30]:
small_train_dataset = torch.utils.data.Subset(
    train_dataset,
    range(64),
)

small_train_loader = DataLoader(
    small_train_dataset,
    batch_size=16,
    shuffle=True,
)

small_val_dataset = torch.utils.data.Subset(
    val_dataset,
    range(64),
)

small_val_loader = DataLoader(
    small_val_dataset,
    batch_size=16,
    shuffle=True,
)

In [34]:
for epoch in range(5):
    train_metrics = run_epoch(model, train_loader, training=True)
    val_metrics = run_epoch(model, val_loader, training=False)

    print(
        f"Epoch {epoch + 1}: "
        f"train loss={train_metrics[0]:.4f}, "
        f"train F1={train_metrics[2]:.4f}, "
        f"train accuracy={train_metrics[1]:.4f},"
        f"val loss={val_metrics[0]:.4f}, "
        f"val F1={val_metrics[2]:.4f},"
        f"val accuracy={val_metrics[1]:.4f}"
    )

Epoch 1: train loss=1.4017, train F1=0.6117, train accuracy=0.6135,val loss=0.5975, val F1=0.8109,val accuracy=0.8150
Epoch 2: train loss=0.5389, train F1=0.8334, train accuracy=0.8327,val loss=0.4766, val F1=0.8394,val accuracy=0.8400
Epoch 3: train loss=0.4711, train F1=0.8467, train accuracy=0.8455,val loss=0.4162, val F1=0.8485,val accuracy=0.8480
Epoch 4: train loss=0.4297, train F1=0.8568, train accuracy=0.8558,val loss=0.3965, val F1=0.8568,val accuracy=0.8550
Epoch 5: train loss=0.4023, train F1=0.8595, train accuracy=0.8582,val loss=0.4093, val F1=0.8454,val accuracy=0.8420
